In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.expand_frame_repr', False)

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [2]:
!pip install -q mlflow dagshub xgboost

import mlflow
import dagshub

dagshub.init(repo_owner='aochi23', repo_name='ml_assn_02', mlflow=True)
mlflow.set_experiment("XGBoost_Training_")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 76.7 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 77.5 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 58.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=8c8d3e47-7783-4e31-b503-9b82a5759247&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=159a1a4c769d463adaf6d92653ef03b0e361e54aef413885ec1754b146854f66




Accessing as aochi23

Initialized MLflow to track repo "aochi23/ml_assn_02"

Repository aochi23/ml_assn_02 initialized!

<Experiment: artifact_location='mlflow-artifacts:/b5fe028c1be44a58b1a0ab41e2adf742', creation_time=1777729799931, experiment_id='2', last_update_time=1777729799931, lifecycle_stage='active', name='XGBoost_Training_', tags={'mlflow.experimentKind': 'custom_model_development'}, trace_location=None, workspace='default'>

# 1. Data Loading


In [50]:
path = "/kaggle/input/competitions/ieee-fraud-detection/"

train_transaction = pd.read_csv(path + "train_transaction.csv")
train_identity    = pd.read_csv(path + "train_identity.csv")


In [51]:
train = train_transaction.merge(train_identity, on="TransactionID", how="left")

print(f"Train shape: {train.shape}")

Train shape: (590540, 434)


In [52]:
fraud_counts = train["isFraud"].value_counts()
fraud_rate   = train["isFraud"].mean() * 100
neg_pos_ratio = int((1 - train["isFraud"].mean()) / train["isFraud"].mean())

print(fraud_counts)
print(f"\nFraud rate: {fraud_rate:.2f}%")
print(f"scale_pos_weight to use: {neg_pos_ratio}")

isFraud
0    569877
1     20663
Name: count, dtype: int64

Fraud rate: 3.50%
scale_pos_weight to use: 27


# 2. Cleaning

In [53]:
y = train['isFraud']
X = train.drop(columns=['isFraud', 'TransactionID'])

In [54]:
X['email_match'] = (X['P_emaildomain'] == X['R_emaildomain']).astype(np.int8)

In [55]:
missing_rate = X.isnull().mean()
high_missing_columns = missing_rate[missing_rate > 0.5].index.tolist()
missing_flags = X[high_missing_columns].isnull().astype(np.int8)

missing_flags.columns = [f"{col}_missing" for col in high_missing_columns]

X = pd.concat([X.drop(columns=high_missing_columns), missing_flags], axis=1).copy()

print(f"Columns dropped: {len(high_missing_columns)}")
print(f"Missing flag columns added: {len(high_missing_columns)}")
print(f"Total columns after: {X.shape[1]}")

Columns dropped: 214
Missing flag columns added: 214
Total columns after: 433


In [56]:
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X.select_dtypes(include=['object']).columns.tolist()

In [57]:
from sklearn.preprocessing import LabelEncoder
label_encoder = LabelEncoder()
for col in cat_cols:
    X[col] = X[col].fillna('unknown')
    X[col] = label_encoder.fit_transform(X[col].astype(str))

print(f"Categorical columns encoded: {len(cat_cols)}")
print(f"Missing values remaining: {X.isnull().sum().sum()}")

Categorical columns encoded: 9
Missing values remaining: 12674972


In [58]:
missing_in_cat = X[cat_cols].isnull().sum().sum()
missing_in_num = X[num_cols].isnull().sum().sum()

print(f"Missing in categorical cols: {missing_in_cat}")
print(f"Missing in numeric cols: {missing_in_num}")

Missing in categorical cols: 0
Missing in numeric cols: 12674972


In [59]:
cols_after_cleaning = X.shape[1]

with mlflow.start_run(run_name="XGB_Cleaning"):
    mlflow.log_params({
        "missing_threshold":         0.5,
        "high_missing_cols_dropped": len(high_missing_columns),
        "missing_flag_cols_added":   len(high_missing_columns),
        "email_match_feature_added": True,
        "label_encoding":            True,
        "cat_cols_encoded":          len(cat_cols),
    })
    mlflow.log_metrics({
        "total_columns_after_cleaning": cols_after_cleaning,
        "total_rows":                   X.shape[0],
    })

🏃 View run XGB_Cleaning at: https://dagshub.com/aochi23/ml_assn_02.mlflow/#/experiments/2/runs/0b4d91f8a1d5460684d6c269e3e1c474
🧪 View experiment at: https://dagshub.com/aochi23/ml_assn_02.mlflow/#/experiments/2


# 3. Feature Engineering

In [60]:
X['hour'] = (X['TransactionDT'] // 3600) % 24
X['day']  = (X['TransactionDT'] // (3600 * 24)) % 7

X['TransactionAmt_log'] = np.log1p(X['TransactionAmt'])
X['TransactionAmt_cents'] = X['TransactionAmt'] % 1
X['amt_is_round'] = (X['TransactionAmt'] % 1 == 0).astype(int)
X['amt_is_round_100'] = (X['TransactionAmt'] % 100 == 0).astype(int)

X['addr_mismatch'] = (X['addr1'] != X['addr2']).astype(int)

In [61]:
X['user_id'] = (
    X['card1'].fillna(-1).astype(str) + '_' +
    X['card2'].fillna(-1).astype(str) + '_' +
    X['addr1'].fillna(-1).astype(str) + '_' +
    X['P_emaildomain'].fillna('unknown').astype(str)
)

X['user_id_count']    = X.groupby('user_id')['user_id'].transform('count')
X['user_id_mean_amt'] = X.groupby('user_id')['TransactionAmt'].transform('mean')
X['user_id_std_amt']  = X.groupby('user_id')['TransactionAmt'].transform('std').fillna(0)
X['user_id_max_amt']  = X.groupby('user_id')['TransactionAmt'].transform('max')

X['user_amt_zscore']   = ((X['TransactionAmt'] - X['user_id_mean_amt']) / X['user_id_std_amt'].replace(0, 1)).clip(-5, 5)
X['user_amt_ratio']    = (X['TransactionAmt'] / X['user_id_mean_amt']).clip(0, 20)
X['user_id_count_log'] = np.log1p(X['user_id_count'])
X['amt_vs_max']        = (X['TransactionAmt'] / X['user_id_max_amt']).clip(0, 1)

X['user_time_delta'] = X.groupby('user_id')['TransactionDT'].transform(lambda x: x.diff().fillna(0))

In [62]:
for col in ['card1', 'card2']:
    X[f'{col}_mean_amt'] = X.groupby(col)['TransactionAmt'].transform('mean')
    X[f'{col}_std_amt']  = X.groupby(col)['TransactionAmt'].transform('std').fillna(0)
    X[f'{col}_count']    = X.groupby(col)[col].transform('count')
    X[f'{col}_amt_zscore'] = (
        (X['TransactionAmt'] - X[f'{col}_mean_amt']) /
        X[f'{col}_std_amt'].replace(0, 1)
    ).clip(-5, 5)

X = X.drop(columns=['user_id'])

print(f"Total features after engineering: {X.shape[1]}")

Total features after engineering: 457


In [63]:
with mlflow.start_run(run_name="XGB_Feature_Engineering"):
    mlflow.log_params({
        "user_id_fields":   "card1_card2_addr1_P_emaildomain",
        "new_features":     "hour,day,TransactionAmt_log,TransactionAmt_cents,amt_is_round,amt_is_round_100,addr_mismatch,user_id_count,user_id_mean_amt,user_id_std_amt,user_id_max_amt,user_amt_zscore,user_amt_ratio,user_id_count_log,amt_vs_max,user_time_delta,card1_aggs,card2_aggs",
        "zscore_clip":      "(-5, 5)",
        "ratio_clip":       "(0, 20)",
        "extra_vs_lr":      "amt_is_round,amt_is_round_100,addr_mismatch,user_id_max_amt,amt_vs_max,user_time_delta,card1_aggs,card2_aggs",
    })
    mlflow.log_metrics({
        "total_features_after_engineering": X.shape[1],
    })

🏃 View run XGB_Feature_Engineering at: https://dagshub.com/aochi23/ml_assn_02.mlflow/#/experiments/2/runs/ee627642e0ca4e89bb2c4ebadf813dfe
🧪 View experiment at: https://dagshub.com/aochi23/ml_assn_02.mlflow/#/experiments/2


# 4. Training

In [64]:
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics import roc_auc_score
from sklearn.decomposition import PCA
from joblib import Parallel, delayed
from xgboost import XGBClassifier
import mlflow.sklearn
import warnings
warnings.filterwarnings("ignore")

In [65]:
class DropIDTransformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None): return self
    def transform(self, X):
        return X.drop(columns=['TransactionID'], errors='ignore').reset_index(drop=True)

In [66]:
class EmailMatchTransformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None): return self
    def transform(self, X):
        X = X.copy()
        if 'P_emaildomain' in X.columns and 'R_emaildomain' in X.columns:
            X['email_match'] = (X['P_emaildomain'] == X['R_emaildomain']).astype(np.int8)
        else:
            X['email_match'] = 0
        return X

In [67]:
class MissingFlagTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.5):
        self.threshold = threshold
        self.high_missing_columns = None

    def fit(self, X, y=None):
        missing_rate = X.isnull().mean()
        self.high_missing_columns = missing_rate[missing_rate > self.threshold].index.tolist()
        return self

    def transform(self, X):
        X = X.copy()
        missing_flags = X[self.high_missing_columns].isnull().astype(np.int8)
        missing_flags.columns = [f"{col}_missing" for col in self.high_missing_columns]
        return pd.concat([X.drop(columns=self.high_missing_columns), missing_flags], axis=1)

In [68]:
class LabelEncodeTransformer(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.encoders = {}
        self.cat_cols = None

    def fit(self, X, y=None):
        self.cat_cols = X.select_dtypes(include=['object']).columns.tolist()
        for col in self.cat_cols:
            le = LabelEncoder()
            le.fit(X[col].fillna('unknown').astype(str))
            self.encoders[col] = le
        return self

    def transform(self, X):
        X = X.copy()
        for col in self.cat_cols:
            X[col] = X[col].fillna('unknown').astype(str)
            known  = set(self.encoders[col].classes_)
            X[col] = X[col].apply(lambda v: v if v in known else 'unknown')
            X[col] = self.encoders[col].transform(X[col])
        return X

In [69]:
class TargetEncodeTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, cols=None, smoothing=10):
        self.cols        = cols or ['card1', 'card2', 'addr1', 'P_emaildomain', 'R_emaildomain']
        self.smoothing   = smoothing
        self.global_mean = None
        self.encoding_map = {}

    def fit(self, X, y):
        X = X.copy()
        y = pd.Series(y).reset_index(drop=True)
        X = X.reset_index(drop=True)
        self.global_mean = y.mean()
        for col in self.cols:
            if col not in X.columns:
                continue
            X[col] = X[col].fillna('unknown').astype(str)
            stats  = pd.DataFrame({'target': y, 'col': X[col]})
            agg    = stats.groupby('col')['target'].agg(['mean', 'count'])
            smooth = (agg['count'] * agg['mean'] + self.smoothing * self.global_mean) / \
                     (agg['count'] + self.smoothing)
            self.encoding_map[col] = smooth
        return self

    def transform(self, X):
        X = X.copy()
        for col in self.cols:
            if col not in X.columns:
                continue
            X[col] = X[col].fillna('unknown').astype(str)
            X[col] = X[col].map(self.encoding_map[col]).fillna(self.global_mean)
        return X

In [70]:
class RichAggregationTransformer(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.card_stats = {}
        self.user_time_stats = None

    def fit(self, X, y=None):
        X = X.copy()
        for col in ['card1', 'card2']:
            self.card_stats[col] = (
                X.groupby(col)['TransactionAmt']
                .agg(['mean', 'std', 'count'])
                .rename(columns={
                    'mean':  f'{col}_mean_amt',
                    'std':   f'{col}_std_amt',
                    'count': f'{col}_count'
                })
            )
        X['user_id'] = (
            X['card1'].fillna(-1).astype(str) + '_' +
            X['card2'].fillna(-1).astype(str) + '_' +
            X['addr1'].fillna(-1).astype(str)
        )
        self.user_time_stats = (
            X.groupby('user_id')['TransactionDT']
            .agg(['mean', 'std'])
            .rename(columns={'mean': 'user_dt_mean', 'std': 'user_dt_std'})
        )
        return self

    def transform(self, X):
        X = X.copy()
        for col in ['card1', 'card2']:
            X = X.join(self.card_stats[col], on=col)
            X[f'{col}_std_amt'] = X[f'{col}_std_amt'].fillna(0)
            X[f'{col}_amt_zscore'] = (
                (X['TransactionAmt'] - X[f'{col}_mean_amt']) /
                X[f'{col}_std_amt'].replace(0, 1)
            ).clip(-5, 5)
        X['user_id'] = (
            X['card1'].fillna(-1).astype(str) + '_' +
            X['card2'].fillna(-1).astype(str) + '_' +
            X['addr1'].fillna(-1).astype(str)
        )
        X = X.join(self.user_time_stats, on='user_id')
        X['user_dt_std'] = X['user_dt_std'].fillna(0)
        X['amt_is_round']     = (X['TransactionAmt'] % 1 == 0).astype(int)
        X['amt_is_round_100'] = (X['TransactionAmt'] % 100 == 0).astype(int)
        X['addr_mismatch']    = (X['addr1'] != X['addr2']).astype(int)
        X = X.drop(columns=['user_id'])
        return X

In [71]:
class VColumnPCATransformer(BaseEstimator, TransformerMixin):
    def __init__(self, n_components=50):
        self.n_components = n_components
        self.pca      = None
        self.imputer  = None
        self.v_cols   = None

    def fit(self, X, y=None):
        X = pd.DataFrame(X)
        self.v_cols  = [c for c in X.columns if str(c).startswith('V')]
        self.imputer = SimpleImputer(strategy='median')
        self.pca     = PCA(n_components=self.n_components, random_state=42)
        v_data       = self.imputer.fit_transform(X[self.v_cols])
        self.pca.fit(v_data)
        return self

    def transform(self, X):
        X      = X.copy()
        v_data = self.imputer.transform(X[self.v_cols])
        pca_df = pd.DataFrame(
            self.pca.transform(v_data),
            columns=[f'V_pca_{i}' for i in range(self.n_components)],
            index=X.index
        )
        X = X.drop(columns=self.v_cols)
        X = pd.concat([X, pca_df], axis=1)
        return X

In [72]:
class FeatureEngineeringTransformer(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.user_stats = None

    def fit(self, X, y=None):
        X = X.copy()
        X['user_id'] = (
            X['card1'].fillna(-1).astype(str) + '_' +
            X['card2'].fillna(-1).astype(str) + '_' +
            X['addr1'].fillna(-1).astype(str) + '_' +
            X['P_emaildomain'].fillna('unknown').astype(str)
        )
        self.user_stats = (
            X.groupby('user_id')['TransactionAmt']
            .agg(['count', 'mean', 'std', 'max'])
            .rename(columns={
                'count': 'uid_count',
                'mean':  'uid_mean',
                'std':   'uid_std',
                'max':   'uid_max'
            })
        )
        return self

    def transform(self, X):
        X = X.copy()
        X['hour'] = (X['TransactionDT'] // 3600) % 24
        X['day']  = (X['TransactionDT'] // (3600 * 24)) % 7
        X['TransactionAmt_log']   = np.log1p(X['TransactionAmt'])
        X['TransactionAmt_cents'] = X['TransactionAmt'] % 1
        X['user_id'] = (
            X['card1'].fillna(-1).astype(str) + '_' +
            X['card2'].fillna(-1).astype(str) + '_' +
            X['addr1'].fillna(-1).astype(str) + '_' +
            X['P_emaildomain'].fillna('unknown').astype(str)
        )
        X = X.join(self.user_stats, on='user_id')
        X = X.rename(columns={
            'uid_count': 'user_id_count',
            'uid_mean':  'user_id_mean_amt',
            'uid_std':   'user_id_std_amt',
            'uid_max':   'user_id_max_amt'
        })
        X['user_id_std_amt']   = X['user_id_std_amt'].fillna(0)
        X['user_amt_zscore']   = ((X['TransactionAmt'] - X['user_id_mean_amt']) / X['user_id_std_amt'].replace(0, 1)).clip(-5, 5)
        X['user_amt_ratio']    = (X['TransactionAmt'] / X['user_id_mean_amt']).clip(0, 20)
        X['user_id_count_log'] = np.log1p(X['user_id_count'])
        X['amt_vs_max']        = (X['TransactionAmt'] / X['user_id_max_amt']).clip(0, 1)
        X['user_time_delta']   = X.groupby('user_id')['TransactionDT'].transform(lambda x: x.diff().fillna(0))
        X = X.drop(columns=['user_id'])
        return X

In [73]:
X_raw = train_transaction.merge(train_identity, on="TransactionID", how="left")
y = X_raw['isFraud']
X_raw = X_raw.drop(columns=['isFraud'])

In [74]:
X_train_raw, X_val_raw, y_train, y_val = train_test_split(
    X_raw, y, test_size=0.2, random_state=42, stratify=y
)

skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

## 4a. Feature Selection Comparison

In [75]:
base_xgb = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=neg_pos_ratio,
    eval_metric='auc',
    random_state=42,
    n_jobs=2
)

In [76]:
pipeline_no_selection = Pipeline([
    ('drop_id',    DropIDTransformer()),
    ('email',      EmailMatchTransformer()),
    ('missing',    MissingFlagTransformer(threshold=0.5)),
    ('rich_agg',   RichAggregationTransformer()),
    ('target_enc', TargetEncodeTransformer()),
    ('encode',     LabelEncodeTransformer()),
    ('features',   FeatureEngineeringTransformer()),
    ('vpca',       VColumnPCATransformer(n_components=50)),
    ('model',      base_xgb)
])

In [77]:
pipeline_no_selection.fit(X_train_raw, y_train)
y_pred = pipeline_no_selection.predict_proba(X_val_raw)[:, 1]
auc = roc_auc_score(y_val, y_pred)
print(f"XGB_Selection_None: AUC = {auc:.4f}")

XGB_Selection_None: AUC = 0.9516


In [78]:
with mlflow.start_run(run_name="XGB_Feature_Selection"):
    mlflow.log_param("strategy", "no_selection_xgb_handles_internally")
    mlflow.log_metric("val_auc", auc)

🏃 View run XGB_Feature_Selection at: https://dagshub.com/aochi23/ml_assn_02.mlflow/#/experiments/2/runs/aafbc298d6c74742997542aecc83da8c
🧪 View experiment at: https://dagshub.com/aochi23/ml_assn_02.mlflow/#/experiments/2


## 4b. Hyperparameter Tuning

In [85]:
param_grid_xgb = {
    'model__max_depth':        [8, 10],
    'model__learning_rate':    [0.05],
    'model__subsample':        [0.8, 0.9],
    'model__colsample_bytree': [0.8],
    'model__reg_alpha':        [0.5],
    'model__reg_lambda':       [2],
    'model__n_estimators':     [500, 700],
}

In [86]:
def run_and_log(run_name, pipeline, param_grid):
    with mlflow.start_run(run_name=run_name):
        grid = GridSearchCV(
            pipeline, param_grid, cv=skf,
            scoring='roc_auc', n_jobs=2,
            return_train_score=True,
            verbose=3
        )
        grid.fit(X_train_raw, y_train)

        for i, params in enumerate(grid.cv_results_['params']):
            run_label = "_".join([
                f"depth{params['model__max_depth']}",
                f"lr{params['model__learning_rate']}",
                f"sub{params['model__subsample']}",
                f"n{params['model__n_estimators']}",
            ])
            with mlflow.start_run(run_name=f"XGB_CV_{run_label}", nested=True):
                mlflow.log_params(params)
                mlflow.log_metric("cv_auc_mean",       grid.cv_results_['mean_test_score'][i])
                mlflow.log_metric("cv_auc_std",        grid.cv_results_['std_test_score'][i])
                mlflow.log_metric("cv_train_auc_mean", grid.cv_results_['mean_train_score'][i])

        best_pipeline = grid.best_estimator_
        train_auc = roc_auc_score(y_train, best_pipeline.predict_proba(X_train_raw)[:, 1])
        val_auc   = roc_auc_score(y_val,   best_pipeline.predict_proba(X_val_raw)[:, 1])

        mlflow.log_params(grid.best_params_)
        mlflow.log_metric("cv_auc_mean",  grid.best_score_)
        mlflow.log_metric("cv_auc_std",   grid.cv_results_['std_test_score'][grid.best_index_])
        mlflow.log_metric("train_auc",    train_auc)
        mlflow.log_metric("val_auc",      val_auc)
        mlflow.log_metric("overfit_gap",  train_auc - val_auc)

        print(f"{run_name} | CV: {grid.best_score_:.4f} | Train: {train_auc:.4f} | Val: {val_auc:.4f} | Gap: {train_auc - val_auc:.4f}")
    return grid

In [87]:
grid_xgb = run_and_log("XGB_CV", pipeline_no_selection, param_grid_xgb)

Fitting 3 folds for each of 8 candidates, totalling 24 fits
[CV 2/3] END model__colsample_bytree=0.8, model__learning_rate=0.05, model__max_depth=8, model__n_estimators=500, model__reg_alpha=0.5, model__reg_lambda=2, model__subsample=0.8;, score=(train=0.999, test=0.940) total time= 3.3min
[CV 3/3] END model__colsample_bytree=0.8, model__learning_rate=0.05, model__max_depth=8, model__n_estimators=500, model__reg_alpha=0.5, model__reg_lambda=2, model__subsample=0.8;, score=(train=0.999, test=0.941) total time= 3.4min
[CV 3/3] END model__colsample_bytree=0.8, model__learning_rate=0.05, model__max_depth=8, model__n_estimators=500, model__reg_alpha=0.5, model__reg_lambda=2, model__subsample=0.9;, score=(train=0.999, test=0.942) total time= 3.4min
[CV 1/3] END model__colsample_bytree=0.8, model__learning_rate=0.05, model__max_depth=8, model__n_estimators=700, model__reg_alpha=0.5, model__reg_lambda=2, model__subsample=0.8;, score=(train=1.000, test=0.941) total time= 4.1min
[CV 3/3] END mod

In [88]:
best_params = {k.replace("model__", ""): v for k, v in grid_xgb.best_params_.items()}
print(f"Best params: {best_params}")

Best params: {'colsample_bytree': 0.8, 'learning_rate': 0.05, 'max_depth': 10, 'n_estimators': 700, 'reg_alpha': 0.5, 'reg_lambda': 2, 'subsample': 0.9}


In [89]:
val_auc_cv = roc_auc_score(y_val, grid_xgb.best_estimator_.predict_proba(X_val_raw)[:, 1])
print(f"Val AUC: {val_auc_cv:.4f}")

Val AUC: 0.9625


In [90]:
best_xgb = XGBClassifier(
    scale_pos_weight=neg_pos_ratio,
    eval_metric='auc',
    random_state=42,
    n_jobs=2,
    **best_params
)

In [92]:
full_pipeline_xgb = Pipeline([
    ('drop_id',    DropIDTransformer()),
    ('email',      EmailMatchTransformer()),
    ('missing',    MissingFlagTransformer(threshold=0.5)),
    ('rich_agg',   RichAggregationTransformer()),
    ('target_enc', TargetEncodeTransformer()),
    ('encode',     LabelEncodeTransformer()),
    ('features',   FeatureEngineeringTransformer()),
    ('vpca',       VColumnPCATransformer(n_components=50)),
    ('model',      best_xgb)
])

In [94]:
full_pipeline_xgb.fit(X_train_raw, y_train)
val_auc = roc_auc_score(y_val, full_pipeline_xgb.predict_proba(X_val_raw)[:, 1])

In [96]:
print(f"Final XGB Val AUC: {val_auc:.4f}")

Final XGB Val AUC: 0.9625


In [97]:
with mlflow.start_run(run_name="XGB_Final_Pipeline"):
    mlflow.log_params({
        "scale_pos_weight":  neg_pos_ratio,
        "missing_threshold": 0.5,
        "vpca_components":   50,
        **best_params
    })
    mlflow.log_metric("val_auc", val_auc)
    mlflow.sklearn.log_model(
        full_pipeline_xgb,
        artifact_path="full_pipeline_xgb",
        registered_model_name="XGB_FullPipeline"
    )
    print(f"Final XGB pipeline VAL AUC: {val_auc:.4f}")

2026/05/02 15:54:37 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/02 15:54:38 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'XGB_FullPipeline'.
2026/05/02 15:54:55 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: XGB_FullPipeline, version 1
Created version '1' of model 'XGB_FullPipeline'.


Final XGB pipeline VAL AUC: 0.9625
🏃 View run XGB_Final_Pipeline at: https://dagshub.com/aochi23/ml_assn_02.mlflow/#/experiments/2/runs/b0d9bb1a0f1d41ccb4f5bcd017c9c45f
🧪 View experiment at: https://dagshub.com/aochi23/ml_assn_02.mlflow/#/experiments/2


In [3]:
model = mlflow.pyfunc.load_model("models:/XGB_FullPipeline/1")

In [4]:
sk_model = model._model_impl.sklearn_model

print(sk_model)
for k, v in sk_model.get_params().items():
    print(k, ":", v)

Pipeline(steps=[('drop_id', DropIDTransformer()),
                ('email', EmailMatchTransformer()),
                ('missing', MissingFlagTransformer()),
                ('rich_agg', RichAggregationTransformer()),
                ('target_enc',
                 TargetEncodeTransformer(cols=['card1', 'card2', 'addr1',
                                               'P_emaildomain',
                                               'R_emaildomain'])),
                ('encode', LabelEncodeTransformer()),
                ('features', FeatureEngineeringTransformer()),
                ('vpc...
                               feature_types=None, feature_weights=None,
                               gamma=None, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=0.05,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_ste

In [100]:
clf = sk_model.named_steps[list(sk_model.named_steps.keys())[-1]]
clf.get_params()

{'objective': 'binary:logistic',
 'base_score': None,
 'booster': None,
 'callbacks': None,
 'colsample_bylevel': None,
 'colsample_bynode': None,
 'colsample_bytree': 0.8,
 'device': None,
 'early_stopping_rounds': None,
 'enable_categorical': False,
 'eval_metric': 'auc',
 'feature_types': None,
 'feature_weights': None,
 'gamma': None,
 'grow_policy': None,
 'importance_type': None,
 'interaction_constraints': None,
 'learning_rate': 0.05,
 'max_bin': None,
 'max_cat_threshold': None,
 'max_cat_to_onehot': None,
 'max_delta_step': None,
 'max_depth': 10,
 'max_leaves': None,
 'min_child_weight': None,
 'missing': nan,
 'monotone_constraints': None,
 'multi_strategy': None,
 'n_estimators': 700,
 'n_jobs': 2,
 'num_parallel_tree': None,
 'random_state': 42,
 'reg_alpha': 0.5,
 'reg_lambda': 2,
 'sampling_method': None,
 'scale_pos_weight': 27,
 'subsample': 0.9,
 'tree_method': None,
 'validate_parameters': None,
 'verbosity': None}

In [5]:
path = "/kaggle/input/competitions/ieee-fraud-detection/"

test_transaction = pd.read_csv(path + "test_transaction.csv")
test_identity    = pd.read_csv(path + "test_identity.csv")

In [6]:
print(test_identity.shape)
print(test_identity.columns[:10])

(141907, 41)
Index(['TransactionID', 'id-01', 'id-02', 'id-03', 'id-04', 'id-05', 'id-06',
       'id-07', 'id-08', 'id-09'],
      dtype='object')


In [7]:
test_df = test_transaction.merge(
    test_identity,
    on="TransactionID",
    how="left"
)

test_df.columns = test_df.columns.str.replace('-', '_')

In [8]:
probs = sk_model.predict_proba(test_df)[:, 1]

In [9]:
submission = pd.DataFrame({
    "TransactionID": test_df["TransactionID"],
    "isFraud": probs
})

In [10]:
submission.head(10)

,TransactionID,isFraud
0,3663549,0.000037
1,3663550,0.001417
2,3663551,0.000019
3,3663552,0.000148
4,3663553,0.000275
5,3663554,0.001611
6,3663555,0.000322
7,3663556,0.006990
8,3663557,0.000024
9,3663558,0.010115


In [11]:
submission.to_csv("xg_boost_submission.csv", index=False)